In [3]:
from dotenv import load_dotenv
load_dotenv()

True

### node-style

In [4]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    age: int=99

In [ ]:
from langchain.agents.middleware import before_model

@before_model
def log_before_model(state, runtime):
    print("-" * 30)
    print("state:", state)
    print("runtime", runtime)
    print("-" * 30)
    return None

In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[],
    middleware=[log_before_model],
    context_schema=Context
)

In [13]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남")
)

----------
state: {'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='09a4cf2f-4b88-4e1a-9284-65e479520c93')]}
runtime Runtime(context=Context(user_name='김일남', age=99), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x00000238E7AB7F60>, previous=None)
----------


{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='09a4cf2f-4b88-4e1a-9284-65e479520c93'),
  AIMessage(content='저는 인공지능이라서 당신의 이름을 알 수 없습니다. 저는 개인 정보를 가지고 있지 않아요.\n\n만약 저에게 당신을 부를 이름을 알려주시면 그렇게 불러드릴 수 있습니다. 😊', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c1868-ba2b-7470-8c04-cdf32ec7f898-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 6, 'output_tokens': 425, 'total_tokens': 431, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 383}})]}

### wrap-style

In [ ]:
# from langchain.agents.middleware import wrap_model_call

# @wrap_model_call
# def inject_user_name(request, handler):
#     print(f"request: {request}")
#     print("-" * 10)
#     return handler(request)

In [7]:
from langchain.agents.middleware import wrap_model_call

@wrap_model_call
def inject_user_name(request, handler):
    print(f"request: {request}")
    print("-" * 10)

    user_name = request.runtime.context.user_name

    if user_name:
        sys_prompt = f"사용자의 이름은 {user_name}입니다."
    else:
        sys_prompt = "사용자의 이름은 알려지지 않았습니다."

    request = request.override(system_prompt = sys_prompt)

    return handler(request)

In [10]:
from langchain.agents.middleware import after_model

@after_model
def log_after_model(state, runtime):
    print(f"after_model_state: {state}")
    print("-" * 10)
    return None

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    middleware=[inject_user_name, log_after_model],
    context_schema=Context
)

In [12]:
agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐야?"}]},
    context=Context(user_name="김일남")
)

request: ModelRequest(model=ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x00000215A5ECB500>, default_metadata=(), model_kwargs={}), messages=[HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='19caa88c-d217-477d-a119-31265cc7d475')], system_message=None, tool_choice=None, tools=[], response_format=None, state={'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='19caa88c-d217-477d-a119-31265cc7d475')]}, runtime=Runtime(context=Context(user_name='김일남', age=99

{'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='19caa88c-d217-477d-a119-31265cc7d475'),
  AIMessage(content='김일남입니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c36f2-8aa4-7c73-baa9-2fefefe8736e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 39, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 34}})]}